# Fase 5 — Avaliação e IA Responsável

**Projeto**: Mineração de Padrões Frequentes em Acidentes de Rodovias Federais  
**Grupo 18**: Erik Roberto Reis Neves, Gabriel Campos Prudente, Felipe Silva Fraga Damasceno  
**Disciplina**: Mineração de Dados — UFMG  

---

## Objetivos desta fase
1. Avaliar quantitativamente e de forma cruzada as regras geradas (foco em *Lift* e *Conviction*)
2. Garantir **Explicabilidade**: Traduzir estatísticas complexas e conjuntos abstratos em afirmações naturais compreensíveis pelo usuário final.
3. Prevenir más interpretações documentando expressamente os limites da correlação (Causalidade vs Co-ocorrência).
4. Documentar o monitoramento do modelo: quais regras são perenes e quais dependem da época do ano?
5. Discorrer sobre vieses ocultos na base da Polícia Rodoviária Federal.

**Alinhamento CRISP-DM**: *Evaluation*


## Setup — Importações e Configurações

In [1]:
"""
Fase 5 -- Avaliacao e IA Responsavel (script executavel)
Avaliacao qualitativa das regras extraidas, traducao para linguagem natural e aplicacao de diretrizes de IA Responsavel.
"""
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Configuracoes

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', None)

PROCESSED_DIR = Path('../data/processed/')
OUTPUT_DIR = Path('../outputs/')
DADOS_DIR = OUTPUT_DIR / 'dados'
TABELAS_DIR = OUTPUT_DIR / 'tabelas'

print('[OK] Imports e configuracoes realizados.')

[OK] Imports e configuracoes realizados.


## 5.0 Carregamento Das Regras

In [3]:
print('\n' + '='*60)
print('5.0 CARREGAMENTO DAS REGRAS')
print('='*60)

regras_globais = pd.read_csv(DADOS_DIR / 'regras_completas.csv')

with open(PROCESSED_DIR / 'rules_por_cluster.pkl', 'rb') as f:
    rules_por_cluster = pickle.load(f)

with open(PROCESSED_DIR / 'rules_por_periodo.pkl', 'rb') as f:
    rules_por_periodo = pickle.load(f)

print(f'Regras Globais: {len(regras_globais):,} regras')
print(f'Clusters mapeados: {list(rules_por_cluster.keys())}')
print(f'Periodos mapeados: {list(rules_por_periodo.keys())}')


5.0 CARREGAMENTO DAS REGRAS


Regras Globais: 7,923 regras
Clusters mapeados: [0, 1]
Periodos mapeados: ['Mes_01', 'Mes_02', 'Mes_03', 'Mes_04']


## 5.1 Avaliação Quantitativa Das Regras

In [4]:
print('\n' + '='*60)
print('5.1 AVALIACAO QUANTITATIVA DAS REGRAS')
print('='*60)

# Top 20 Regras Globais por Lift
print('\n=== Top 20 Regras Globais (ordenadas por Lift) ===')
top_globais = regras_globais.nlargest(20, 'lift')
cols_view = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage']
print(top_globais[cols_view].to_string(index=False))
top_globais.to_csv(TABELAS_DIR / 'top20_regras_globais.csv', index=False)

# Top 10 Regras por Cluster
print('\n=== Top 10 Regras por Cluster (ordenadas por Lift) ===')
for c, rules_c in rules_por_cluster.items():
    print(f'\n--- Cluster {c} ---')
    top_c = rules_c.nlargest(10, 'lift')
    # O dataframe do cluster tem formato diferente do CSV (frozensets originais)
    if 'antecedents_str' not in top_c.columns:
        top_c['antecedents_str'] = top_c['antecedents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
        top_c['consequents_str'] = top_c['consequents'].apply(lambda x: ' & '.join(sorted([str(i) for i in x])))
    print(top_c[cols_view].to_string(index=False))
    top_c.to_csv(TABELAS_DIR / f'top10_regras_cluster_{c}.csv', index=False)


5.1 AVALIACAO QUANTITATIVA DAS REGRAS

=== Top 20 Regras Globais (ordenadas por Lift) ===
                                                                     antecedents_str                                                               consequents_str  support  confidence    lift  leverage
                                           classificacao_acidente_Com Vítimas Fatais                                        gravidade_binaria_Fatal & uso_solo_Não   0.0543      0.7678 14.1469    0.0504
                                                             gravidade_binaria_Fatal                      classificacao_acidente_Com Vítimas Fatais & uso_solo_Não   0.0543      0.7678 14.1469    0.0504
                                                             gravidade_binaria_Fatal                                     classificacao_acidente_Com Vítimas Fatais   0.0707      1.0000 14.1469    0.0657
                            classificacao_acidente_Com Vítimas Fatais & uso_solo_Não                 

## 5.2 Diretriz 1: Explicabilidade (Traducao Para Linguagem Natural)

In [5]:
print('\n' + '='*60)
print('5.2 DIRETRIZ 1: EXPLICABILIDADE (TRADUCAO PARA LINGUAGEM NATURAL)')
print('='*60)

# 1. Filtragem para explicabilidade (<= 3 antecedentes)
regras_explicaveis = regras_globais[regras_globais['n_antecedents'] <= 3].copy()
print(f'Regras com ate 3 antecedentes: {len(regras_explicaveis):,} de {len(regras_globais):,}')

# 2. Funcao de traducao
def regra_para_texto(row):
    """Converte uma regra de associacao para linguagem natural, limpando os prefixos."""
    def limpar_item(item_str):
        # Remove os prefixos como "causa_acidente_", "tipo_acidente_", etc.
        if item_str.startswith('gravidade_binaria_'):
            estado = item_str.replace('gravidade_binaria_', '')
            return f"Acidente {estado}"
        elif item_str.startswith('fim_de_semana_'):
            estado = item_str.replace('fim_de_semana_', '')
            return "Em um fim de semana" if estado == "Sim" else "Fora do fim de semana"
        elif '_' in item_str:
            partes = item_str.split('_', 1)
            return partes[1]
        return item_str

    ante = [limpar_item(i.strip()) for i in row['antecedents_str'].split('&')]
    cons = [limpar_item(i.strip()) for i in row['consequents_str'].split('&')]
    
    ante_str = ' E '.join(ante)
    cons_str = ' E '.join(cons)
    
    return (f"Quando {ante_str}, então {cons_str}. "
            f"[Suporte: {row['support']:.1%} | Confiança: {row['confidence']:.1%} | Lift: {row['lift']:.1f}]")

regras_explicaveis['explicacao_natural'] = regras_explicaveis.apply(regra_para_texto, axis=1)

print('\n=== Top 15 Regras Traduzidas para Linguagem Natural ===')
top_traduzidas = regras_explicaveis.nlargest(15, 'lift')
for i, txt in enumerate(top_traduzidas['explicacao_natural']):
    print(f'{i+1}. {txt}')

top_traduzidas[['explicacao_natural']].to_csv(TABELAS_DIR / 'top15_regras_traduzidas.csv', index=False)

print('\n--- DISCLAIMER DE CAUSALIDADE ---')
print('AVISO IMPORTANTE: As regras de associacao extraidas indicam apenas *CO-OCORRENCIA* estatistica ')
print('baseada na frequencia dos dados historicos. Uma regra do tipo A -> B significa que B e frequentemente ')
print('observado junto com A, mas NAO IMPLICA que A causa B. Decisoes criticas baseadas nestas regras devem ')
print('ser submetidas a uma avaliacao de dominio e analise de confounding factors por especialistas em transito.')


5.2 DIRETRIZ 1: EXPLICABILIDADE (TRADUCAO PARA LINGUAGEM NATURAL)
Regras com ate 3 antecedentes: 5,694 de 7,923



=== Top 15 Regras Traduzidas para Linguagem Natural ===
1. Quando acidente_Com Vítimas Fatais, então Acidente Fatal E solo_Não. [Suporte: 5.4% | Confiança: 76.8% | Lift: 14.1]
2. Quando Acidente Fatal, então acidente_Com Vítimas Fatais E solo_Não. [Suporte: 5.4% | Confiança: 76.8% | Lift: 14.1]
3. Quando Acidente Fatal, então acidente_Com Vítimas Fatais. [Suporte: 7.1% | Confiança: 100.0% | Lift: 14.1]
4. Quando acidente_Com Vítimas Fatais E solo_Não, então Acidente Fatal. [Suporte: 5.4% | Confiança: 100.0% | Lift: 14.1]
5. Quando Acidente Fatal E solo_Não, então acidente_Com Vítimas Fatais. [Suporte: 5.4% | Confiança: 100.0% | Lift: 14.1]
6. Quando acidente_Com Vítimas Fatais, então Acidente Fatal. [Suporte: 7.1% | Confiança: 100.0% | Lift: 14.1]
7. Quando semana_domingo E dia_Pleno dia, então horaria_Tarde E Em um fim de semana E Acidente Não Fatal. [Suporte: 5.4% | Confiança: 58.1% | Lift: 5.8]
8. Quando horaria_Tarde E Em um fim de semana E Acidente Não Fatal, então semana_domingo

## 5.3 Diretriz 2: Monitoramento e Avaliação (Estabilidade Temporal)

In [6]:
print('\n' + '='*60)
print('5.3 DIRETRIZ 2: MONITORAMENTO E AVALIACAO (ESTABILIDADE TEMPORAL)')
print('='*60)

from collections import Counter

# Converter regras para string unica para comparacao entre dicionarios
def regra_key(row):
    # Tratar caso seja frozenset (do pickle) ou string (se ja convertido)
    if isinstance(row['antecedents'], (frozenset, set)):
        ante = ' & '.join(sorted([str(i) for i in row['antecedents']]))
    else:
        ante = row['antecedents']
        
    if isinstance(row['consequents'], (frozenset, set)):
        cons = ' & '.join(sorted([str(i) for i in row['consequents']]))
    else:
        cons = row['consequents']
        
    return (ante, cons)

contagem = Counter()
for periodo, rules_p in rules_por_periodo.items():
    for _, row in rules_p.iterrows():
        contagem[regra_key(row)] += 1

n_periodos = len(rules_por_periodo)
regras_estaveis = {k for k, v in contagem.items() if v >= n_periodos * 0.5}
regras_transitorias = {k for k, v in contagem.items() if v < n_periodos * 0.3}

print(f'Total de periodos avaliados: {n_periodos}')
print(f'Regras estaveis (aparecem em >= {n_periodos*0.5:.0f} periodos): {len(regras_estaveis):,} regras')
print(f'Regras transitorias (aparecem em < {n_periodos*0.3:.0f} periodos): {len(regras_transitorias):,} regras')

print('\nTop 10 Regras Absolutamente Estaveis (aparecem em todos os periodos com alto suporte medio):')
# Vamos pegar as estaveis e calcular a media das metricas para elas
dados_estaveis = []
for regra_tuple in [k for k, v in contagem.items() if v == n_periodos]:
    ante, cons = regra_tuple
    # Procurar essa regra na tabela global
    mask = (regras_globais['antecedents_str'] == ante) & (regras_globais['consequents_str'] == cons)
    match = regras_globais[mask]
    if len(match) > 0:
        dados_estaveis.append(match.iloc[0])

if dados_estaveis:
    df_estaveis = pd.DataFrame(dados_estaveis).sort_values('lift', ascending=False)
    print(df_estaveis[cols_view].head(10).to_string(index=False))
    df_estaveis.to_csv(TABELAS_DIR / 'regras_estaveis.csv', index=False)


5.3 DIRETRIZ 2: MONITORAMENTO E AVALIACAO (ESTABILIDADE TEMPORAL)


Total de periodos avaliados: 4
Regras estaveis (aparecem em >= 2 periodos): 9,021 regras
Regras transitorias (aparecem em < 1 periodos): 12,705 regras

Top 10 Regras Absolutamente Estaveis (aparecem em todos os periodos com alto suporte medio):


                                                       antecedents_str                                                      consequents_str  support  confidence    lift  leverage
                             classificacao_acidente_Com Vítimas Fatais                                              gravidade_binaria_Fatal   0.0707      1.0000 14.1469    0.0657
                                               gravidade_binaria_Fatal                            classificacao_acidente_Com Vítimas Fatais   0.0707      1.0000 14.1469    0.0657
 dia_semana_domingo & fase_dia_Pleno dia & gravidade_binaria_Não Fatal                              faixa_horaria_Tarde & fim_de_semana_Sim   0.0543      0.6136  5.6885    0.0447
                               dia_semana_domingo & fase_dia_Pleno dia                              faixa_horaria_Tarde & fim_de_semana_Sim   0.0573      0.6129  5.6817    0.0472
                              dia_semana_domingo & faixa_horaria_Tarde fase_dia_Pleno dia & fim_de_semana

## 5.4 Análise Crítica e Limitações Do Modelo

In [7]:
print('\n' + '='*60)
print('5.4 ANALISE CRITICA E LIMITACOES DO MODELO')
print('='*60)

print("""
1. VIÉS NOS DADOS (SELECTION BIAS):
O modelo so observa acidentes que foram efetivamente reportados a Policia Rodoviaria Federal. 
Acidentes muito leves sem envolvimento policial nao estao presentes. A generalizacao destas 
regras deve se restringir ao escopo de "Acidentes nas PRFs".

2. LIMITACOES DO METODO:
O FP-Growth opera sobre dados binarios/categoricos. A discretizacao da coluna KM (faixa_km) e de
Horario (faixa_horaria) perdeu as nuancas matematicas continuas. Diferencas marginais de 1km 
podem ter ficado em bins separados abruptamente.

3. CONFUNDIMENTO E FATORES OCULTOS:
O algoritmo nao leva em consideracao volume de trafego. Por exemplo, a regra "Fim de Semana -> 
Acidente tipo X" pode ocorrer apenas porque ha mais carros nas vias aos finais de semana e nao 
por periculosidade inerente da data.
""")
print('\n[OK] Fase 5 de Avaliacao concluida.')


5.4 ANALISE CRITICA E LIMITACOES DO MODELO

1. VIÉS NOS DADOS (SELECTION BIAS):
O modelo so observa acidentes que foram efetivamente reportados a Policia Rodoviaria Federal. 
Acidentes muito leves sem envolvimento policial nao estao presentes. A generalizacao destas 
regras deve se restringir ao escopo de "Acidentes nas PRFs".

2. LIMITACOES DO METODO:
O FP-Growth opera sobre dados binarios/categoricos. A discretizacao da coluna KM (faixa_km) e de
Horario (faixa_horaria) perdeu as nuancas matematicas continuas. Diferencas marginais de 1km 
podem ter ficado em bins separados abruptamente.

3. CONFUNDIMENTO E FATORES OCULTOS:
O algoritmo nao leva em consideracao volume de trafego. Por exemplo, a regra "Fim de Semana -> 
Acidente tipo X" pode ocorrer apenas porque ha mais carros nas vias aos finais de semana e nao 
por periculosidade inerente da data.


[OK] Fase 5 de Avaliacao concluida.
